# TALLIP V2 Master Experimental Suite (Zero-Leakage Revision)
**Manuscript:** *Subword Tokenization Alienation and Cross-Dialect Parameter-Efficient Adaptation Across the Bangla Dialect Continuum* (ACM TALLIP Revision)

### Key Features:
- Runs non-destructively against `data/tallip_v2/tallip_v2_clean_benchmark.csv` (`GroupShuffleSplit` by `Standard_Text` with **0.00% semantic/parallel leakage**, NFC + SHA-256 deduplicated).
- Saves all results, predictions, diagnostics, and 200 Error Taxonomy candidates into `results/tallip_v2/` on Google Drive.
- **Fault-Tolerant Incremental Cache:** Every finished run caches to `results/tallip_v2/cache/`. If Colab disconnects, simply re-run all cells and completed experiments skip automatically!

In [ ]:
# Cell 1: Colab Environment Setup, Dependencies & Reproducibility Seeds
# CRITICAL: Run pip uninstall/install BEFORE importing numpy/pandas/pyarrow to prevent C-ABI mismatch!
!pip uninstall -y datasets pyarrow torchao evaluate -q
!pip install -q --upgrade transformers peft accelerate scikit-learn scipy statsmodels seaborn matplotlib tabulate emoji
!pip install -q git+https://github.com/csebuetnlp/normalizer

import os, sys, gc, time, json, math, random, shutil, inspect, re, unicodedata
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Master Configuration
SEEDS = [42, 101, 2026]   # 3 deterministic seeds for mean/SD & statistical testing
PRIMARY_SEED = 42
REGIONS = ['Chattogram', 'Sylhet', 'Noakhali', 'Mymensingh']
LABEL_NAMES = ['negative', 'neutral', 'positive']
LABEL_TO_ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID_TO_LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}

def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_all_seeds(PRIMARY_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 75)
print(f'PyTorch : {torch.__version__} | Device: {device}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB)')
print('=' * 75)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
normalizer 0.0.1 requires emoji==1.4.2, but you have emoji 2.16.0 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
PyTorch : 2.14.1+cu130 | Device: cuda
GPU     : NVIDIA A100-SXM4-40GB (39.49 GB)


In [ ]:
# Cell 2: Google Drive Mount, Cache Init, BUET Normalizer & Zero-Leakage Check
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    WORKSPACE_ROOT = '/content/drive/MyDrive/Bangla_Dialect_Sentiment_Analysis'
else:
    WORKSPACE_ROOT = os.path.abspath('.')

os.chdir(WORKSPACE_ROOT)
DATA_CSV = os.path.join(WORKSPACE_ROOT, 'data', 'tallip_v2', 'tallip_v2_clean_benchmark.csv')
OUT_DIR = os.path.join(WORKSPACE_ROOT, 'results', 'tallip_v2')
CACHE_DIR = os.path.join(OUT_DIR, 'cache')
PREDS_DIR = os.path.join(OUT_DIR, 'preds')
LOCAL_CKPT_DIR = '/content/tallip_v2_scratch_ckpts' if IN_COLAB else os.path.join(OUT_DIR, 'tmp_ckpts')
LOCAL_S1_DIR = '/content/tallip_v2_stage1_models' if IN_COLAB else os.path.join(OUT_DIR, 'tmp_s1')
for d in [OUT_DIR, CACHE_DIR, PREDS_DIR, LOCAL_CKPT_DIR, LOCAL_S1_DIR]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(DATA_CSV):
    print('Clean V2 dataset not found — running build_clean_v2_dataset.py automatically...')
    import subprocess
    subprocess.run([sys.executable, os.path.join(WORKSPACE_ROOT, 'scripts', 'tallip_v2', 'build_clean_v2_dataset.py')], check=True)

df = pd.read_csv(DATA_CSV, encoding='utf-8')
df['label'] = df['Sentiment'].astype(str).str.lower().map(LABEL_TO_ID)
assert df['label'].isna().sum() == 0

# BUET Normalizer (official preprocessor for BanglaBERT)
try:
    from normalizer import normalize as buet_normalize
    BUET_NORM_AVAILABLE = True
    print('BUET normalizer loaded successfully.')
except Exception as e:
    print(f'WARNING: BUET normalizer unavailable ({e}); falling back to NFC clean.')
    buet_normalize = lambda x: unicodedata.normalize('NFC', str(x))
    BUET_NORM_AVAILABLE = False

def apply_buet_norm(text):
    s = buet_normalize(str(text))
    return s if isinstance(s, str) and s.strip() else str(text)

df['Standard_Text_BUET'] = [apply_buet_norm(t) for t in df['Standard_Text']]
df['Dialect_Text_BUET']  = [apply_buet_norm(t) for t in df['Dialect_Text']]

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

# Verify zero leakage
assert len(set(train_df['group_id']) & set(test_df['group_id'])) == 0
assert len(set(train_df['group_id']) & set(val_df['group_id'])) == 0
assert len(set(val_df['group_id']) & set(test_df['group_id'])) == 0
assert len(set(train_df['Standard_Text']) & set(test_df['Standard_Text'])) == 0
print(f'Verified 0.00% Cross-Split Leakage! Train={len(train_df):,} | Val={len(val_df):,} | Test={len(test_df):,}')
print('Test set pure 4-dialect ANUBHUTI rows:', int((test_df['Is_Pure_4Dialect_Parallel'] == 1).sum()))

Mounted at /content/drive
BUET normalizer loaded successfully.
Verified 0.00% Cross-Split Leakage! Train=13,491 | Val=1,689 | Test=1,685
Test set pure 4-dialect ANUBHUTI rows: 988


In [ ]:
# Cell 3: Multi-Tokenizer Subword Fertility Profiling + Utterance-Level Linguistic Distance
import transformers
import transformers.utils as tf_utils
import transformers.utils.import_utils as tf_iu
tf_iu._datasets_available = False
tf_iu.is_datasets_available = lambda: False
if hasattr(tf_utils, 'is_datasets_available'): tf_utils.is_datasets_available = lambda: False
from transformers import AutoTokenizer
from collections import Counter as PyCounter

BENCHMARK_MODELS = {
    'BanglaBERT': 'csebuetnlp/banglabert',
    'mBERT': 'bert-base-multilingual-cased',
    'XLM-RoBERTa': 'xlm-roberta-base'
}

def norm_levenshtein(a: str, b: str) -> float:
    if a == b: return 0.0
    la, lb = len(a), len(b)
    if la == 0 or lb == 0: return 1.0
    if la > lb: a, b, la, lb = b, a, lb, la
    prev = list(range(la + 1))
    for i, cb in enumerate(b, 1):
        curr = [i]
        for j, ca in enumerate(a, 1):
            curr.append(min(curr[-1] + 1, prev[j] + 1, prev[j-1] + (ca != cb)))
        prev = curr
    return prev[-1] / max(la, lb)

def char_ngram_dissim(a: str, b: str, n=3) -> float:
    if len(a) < n or len(b) < n:
        return norm_levenshtein(a, b)
    ca = PyCounter(a[i:i+n] for i in range(len(a)-n+1))
    cb = PyCounter(b[i:i+n] for i in range(len(b)-n+1))
    inter = sum(min(ca[k], cb[k]) for k in ca.keys() & cb.keys())
    total = sum(ca.values()) + sum(cb.values())
    return 1.0 - (2.0 * inter / max(1, total))

tok_cache_json = os.path.join(CACHE_DIR, 'tokenizer_diagnostics_v2.json')
utt_diag_csv = os.path.join(OUT_DIR, 'utterance_diagnostics.csv')

if os.path.exists(tok_cache_json) and os.path.exists(utt_diag_csv):
    with open(tok_cache_json, 'r', encoding='utf-8') as f: tokenizer_summary = json.load(f)
    utt_df = pd.read_csv(utt_diag_csv, encoding='utf-8')
    print('Loaded cached tokenizer diagnostics & utterance features.')
else:
    tokenizers = {k: AutoTokenizer.from_pretrained(v) for k, v in BENCHMARK_MODELS.items()}
    prep_modes = {
        'raw': ('Standard_Text_Raw', 'Dialect_Text_Raw'),
        'nfc_clean': ('Standard_Text', 'Dialect_Text'),
        'buet_norm': ('Standard_Text_BUET', 'Dialect_Text_BUET')
    }
    tokenizer_summary = {}
    for mode_name, (std_col, dia_col) in prep_modes.items():
        tokenizer_summary[mode_name] = {}
        for m_label, tok in tokenizers.items():
            reg_stats = {}
            for reg in REGIONS:
                sub = df[df['Dialect_Region'] == reg]
                s_tok, d_tok, s_w, d_w = 0, 0, 0, 0
                per_utt_s, per_utt_d = [], []
                for st, dt in zip(sub[std_col], sub[dia_col]):
                    sw, dw = max(1, len(str(st).split())), max(1, len(str(dt).split()))
                    st_len, dt_len = len(tok.tokenize(str(st))), len(tok.tokenize(str(dt)))
                    s_tok += st_len; d_tok += dt_len; s_w += sw; d_w += dw
                    per_utt_s.append(st_len / sw); per_utt_d.append(dt_len / dw)
                # Corpus-weighted fertility + utterance mean fertility
                f_std = s_tok / max(1, s_w)
                f_dia = d_tok / max(1, d_w)
                u_std = float(np.mean(per_utt_s))
                u_dia = float(np.mean(per_utt_d))
                reg_stats[reg] = {
                    'corpus_std_fert': round(f_std, 4),
                    'corpus_dia_fert': round(f_dia, 4),
                    'corpus_delta_pct': round(((f_dia - f_std)/f_std)*100.0, 2),
                    'utt_mean_std_fert': round(u_std, 4),
                    'utt_mean_dia_fert': round(u_dia, 4),
                    'utt_mean_delta_pct': round(((u_dia - u_std)/u_std)*100.0, 2)
                }
            tokenizer_summary[mode_name][m_label] = reg_stats

    with open(tok_cache_json, 'w', encoding='utf-8') as f:
        json.dump(tokenizer_summary, f, indent=2)

    # Utterance-level diagnostics on full test_df (+ all rows if fast, let's compute on full df!)
    rows_diag = []
    tok_bb, tok_mb, tok_xr = tokenizers['BanglaBERT'], tokenizers['mBERT'], tokenizers['XLM-RoBERTa']
    for idx, row in df.iterrows():
        st_b, dt_b = str(row['Standard_Text_BUET']), str(row['Dialect_Text_BUET'])
        dt_raw = str(row['Dialect_Text_Raw'])
        dw = max(1, len(dt_b.split()))
        dw_raw = max(1, len(dt_raw.split()))
        sw = max(1, len(st_b.split()))
        bb_d = len(tok_bb.tokenize(dt_b)) / dw
        bb_s = len(tok_bb.tokenize(st_b)) / sw
        bb_raw_d = len(tok_bb.tokenize(dt_raw)) / dw_raw
        mb_d = len(tok_mb.tokenize(dt_b)) / dw
        xr_d = len(tok_xr.tokenize(dt_b)) / dw
        rows_diag.append({
            'sample_id': row['sample_id'],
            'split': row['split'],
            'Dialect_Region': row['Dialect_Region'],
            'Source': row['Source'],
            'Sentiment': row['Sentiment'],
            'label': int(row['label']),
            'Is_Pure_4Dialect_Parallel': int(row['Is_Pure_4Dialect_Parallel']),
            'Is_Pure_Surprise': int(row['Is_Pure_Surprise']),
            'word_count': dw,
            'char_len': len(dt_b),
            'norm_edit_dist': round(norm_levenshtein(st_b, dt_b), 4),
            'char3_dissim': round(char_ngram_dissim(st_b, dt_b, 3), 4),
            'fert_bb_buet': round(bb_d, 4),
            'fert_bb_std_buet': round(bb_s, 4),
            'fert_bb_delta': round(bb_d - bb_s, 4),
            'fert_bb_raw': round(bb_raw_d, 4),
            'fert_mb_buet': round(mb_d, 4),
            'fert_xlmr_buet': round(xr_d, 4)
        })
    utt_df = pd.DataFrame(rows_diag)
    utt_df.to_csv(utt_diag_csv, index=False, encoding='utf-8')
    print('Saved utterance_diagnostics.csv & tokenizer_diagnostics_v2.json')

print('\nTokenizer Summary (BUET Normalized vs Raw - Corpus Fertility Delta %):')
for mode in ['raw', 'buet_norm']:
    print(f'[{mode}]')
    for m in ['BanglaBERT', 'mBERT', 'XLM-RoBERTa']:
        line = ' | '.join(f'{r}: {tokenizer_summary[mode][m][r]["corpus_std_fert"]:.3f}->{tokenizer_summary[mode][m][r]["corpus_dia_fert"]:.3f} ({tokenizer_summary[mode][m][r]["corpus_delta_pct"]:+.2f}%)' for r in REGIONS)
        print(f'  {m:<11}: {line}')

Loaded cached tokenizer diagnostics & utterance features.

Tokenizer Summary (BUET Normalized vs Raw - Corpus Fertility Delta %):
[raw]
  BanglaBERT : Chattogram: 1.177->1.625 (+38.12%) | Sylhet: 1.238->1.516 (+22.45%) | Noakhali: 1.238->1.488 (+20.16%) | Mymensingh: 1.238->1.436 (+15.97%)
  mBERT      : Chattogram: 2.412->2.741 (+13.63%) | Sylhet: 2.563->2.803 (+9.37%) | Noakhali: 2.563->2.843 (+10.93%) | Mymensingh: 2.562->2.830 (+10.44%)
  XLM-RoBERTa: Chattogram: 1.732->2.425 (+39.98%) | Sylhet: 1.821->2.181 (+19.80%) | Noakhali: 1.820->2.178 (+19.66%) | Mymensingh: 1.820->2.154 (+18.35%)
[buet_norm]
  BanglaBERT : Chattogram: 1.170->1.739 (+48.61%) | Sylhet: 1.224->1.523 (+24.45%) | Noakhali: 1.224->1.502 (+22.70%) | Mymensingh: 1.224->1.425 (+16.46%)
  mBERT      : Chattogram: 2.535->3.025 (+19.36%) | Sylhet: 2.678->2.924 (+9.19%) | Noakhali: 2.677->2.966 (+10.79%) | Mymensingh: 2.677->2.931 (+9.47%)
  XLM-RoBERTa: Chattogram: 1.724->2.421 (+40.45%) | Sylhet: 1.809->2.171 (+20.03

In [ ]:
# Cell 4: Universal Trainer, Full Per-Class & Slice Evaluator, McNemar & Cache Helpers
import transformers.utils as tf_utils
import transformers.utils.import_utils as tf_iu
tf_iu._datasets_available = False
tf_iu.is_datasets_available = lambda: False
if hasattr(tf_utils, 'is_datasets_available'): tf_utils.is_datasets_available = lambda: False
if hasattr(tf_iu, 'is_torchao_available'): tf_iu.is_torchao_available = lambda: False
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from torch.utils.data import Dataset as TorchDataset
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from scipy.stats import binomtest
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
    peft.import_utils.is_auto_gptq_available = lambda: False
except Exception: pass
from peft import LoraConfig, get_peft_model, TaskType

class TextDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = [str(t) for t in texts]
        self.labels = [int(y) for y in labels]
        self.tok = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], truncation=True, max_length=self.max_len, padding=False)
        enc['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return enc

class ClassWeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        valid_params = inspect.signature(Trainer.__init__).parameters
        if 'tokenizer' in kwargs and 'tokenizer' not in valid_params:
            tok = kwargs.pop('tokenizer')
            if 'processing_class' in valid_params: kwargs['processing_class'] = tok
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        if self.class_weights is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(model.device))
        else:
            loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_class_weights(sub_df):
    tot = len(sub_df)
    cnts = sub_df['label'].value_counts().to_dict()
    weights = [tot / (3.0 * max(1, cnts.get(c, 1))) for c in range(3)]
    return torch.tensor(weights, dtype=torch.float), weights

def build_training_args(out_dir, num_epochs=3, batch_size=16, lr=2e-5, seed=42, save_best=True):
    valid = inspect.signature(TrainingArguments.__init__).parameters
    kw = {
        'output_dir': out_dir,
        'num_train_epochs': num_epochs,
        'per_device_train_batch_size': batch_size,
        'per_device_eval_batch_size': batch_size * 2,
        'learning_rate': lr,
        'weight_decay': 0.01,
        'logging_steps': 50,
        'save_strategy': 'epoch' if save_best else 'no',
        'save_total_limit': 1,
        'load_best_model_at_end': save_best,
        'metric_for_best_model': 'macro_f1',
        'greater_is_better': True,
        'report_to': 'none',
        'seed': seed,
        'data_seed': seed
    }
    if save_best:
        if 'eval_strategy' in valid: kw['eval_strategy'] = 'epoch'
        elif 'evaluation_strategy' in valid: kw['evaluation_strategy'] = 'epoch'
    if 'warmup_ratio' in valid: kw['warmup_ratio'] = 0.06
    elif 'warmup_steps' in valid: kw['warmup_steps'] = 50
    return TrainingArguments(**{k: v for k, v in kw.items() if k in valid})

def eval_metrics_fn(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'macro_f1': float(f1_score(labels, preds, average='macro', zero_division=0))
    }

def bootstrap_ci_both(y_true, y_pred, n_boot=1000, seed=42):
    rng = np.random.RandomState(seed)
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    n = len(yt)
    accs, f1s = [], []
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        accs.append(float(np.mean(yt[idx] == yp[idx])))
        f1s.append(float(f1_score(yt[idx], yp[idx], average='macro', zero_division=0)))
    return (
        [round(float(np.percentile(accs, 2.5)), 4), round(float(np.percentile(accs, 97.5)), 4)],
        [round(float(np.percentile(f1s, 2.5)), 4), round(float(np.percentile(f1s, 97.5)), 4)]
    )

def full_slice_report(y_true, y_pred):
    yt, yp = np.asarray(y_true), np.asarray(y_pred)
    acc = float(accuracy_score(yt, yp))
    mf1 = float(f1_score(yt, yp, average='macro', zero_division=0))
    wf1 = float(f1_score(yt, yp, average='weighted', zero_division=0))
    acc_ci, f1_ci = bootstrap_ci_both(yt, yp)
    p, r, f1, sup = precision_recall_fscore_support(yt, yp, labels=[0,1,2], zero_division=0)
    per_class = {}
    for i, cname in enumerate(LABEL_NAMES):
        per_class[cname] = {
            'precision': round(float(p[i]), 4),
            'recall': round(float(r[i]), 4),
            'f1': round(float(f1[i]), 4),
            'support': int(sup[i])
        }
    return {
        'n': int(len(yt)),
        'accuracy': round(acc, 4),
        'acc_ci_95': acc_ci,
        'macro_f1': round(mf1, 4),
        'f1_ci_95': f1_ci,
        'weighted_f1': round(wf1, 4),
        'macro_precision': round(float(np.mean(p)), 4),
        'macro_recall': round(float(np.mean(r)), 4),
        'per_class': per_class
    }

def evaluate_test_comprehensive(trainer, tok, text_col, max_len=128, pred_tag=None):
    test_ds = TextDataset(test_df[text_col].tolist(), test_df['label'].tolist(), tok, max_len=max_len)
    out = trainer.predict(test_ds)
    preds = np.argmax(out.predictions, axis=-1)
    y_true = test_df['label'].values
    if pred_tag:
        np.save(os.path.join(PREDS_DIR, f'{pred_tag}.npy'), preds)
    res = {'pooled': full_slice_report(y_true, preds), 'per_dialect': {}}
    for reg in REGIONS:
        mask = (test_df['Dialect_Region'].values == reg)
        res['per_dialect'][reg] = full_slice_report(y_true[mask], preds[mask])
    # Pure 4-dialect parallel ANUBHUTI slice
    par_mask = (test_df['Is_Pure_4Dialect_Parallel'].values == 1)
    res['anubhuti_4way_pooled'] = full_slice_report(y_true[par_mask], preds[par_mask])
    res['anubhuti_4way_per_dialect'] = {}
    for reg in REGIONS:
        m = par_mask & (test_df['Dialect_Region'].values == reg)
        res['anubhuti_4way_per_dialect'][reg] = full_slice_report(y_true[m], preds[m])
    # Sensitivity excluding 29 pure-Surprise ANUBHUTI rows
    ns_mask = (test_df['Is_Pure_Surprise'].values == 0)
    res['no_pure_surprise_pooled'] = full_slice_report(y_true[ns_mask], preds[ns_mask])
    return res, preds

def mcnemar_vs(y_true, preds_a, preds_b):
    ca = (np.asarray(preds_a) == np.asarray(y_true))
    cb = (np.asarray(preds_b) == np.asarray(y_true))
    n01 = int(np.sum((~ca) & cb))  # B fixes A error
    n10 = int(np.sum(ca & (~cb)))  # B breaks A hit
    n_disc = n01 + n10
    pval = float(binomtest(min(n01, n10), n_disc, 0.5).pvalue) if n_disc > 0 else 1.0
    return {'b_fixes_a': n01, 'b_breaks_a': n10, 'net_gain': n01 - n10, 'p_value': round(pval, 6)}

def count_param_breakdown(model):
    total = sum(p.numel() for p in model.parameters())
    lora_p = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and 'lora_' in n)
    head_p = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and ('classifier' in n or 'score' in n))
    other_p = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and 'lora_' not in n and 'classifier' not in n and 'score' not in n)
    train_tot = lora_p + head_p + other_p
    return {
        'total_params': int(total),
        'trainable_total': int(train_tot),
        'trainable_pct': round(train_tot / total * 100.0, 4),
        'lora_only_params': int(lora_p),
        'lora_only_pct': round(lora_p / total * 100.0, 4),
        'head_params': int(head_p),
        'head_pct': round(head_p / total * 100.0, 4)
    }

def run_cached(exp_key, fn):
    cpath = os.path.join(CACHE_DIR, f'{exp_key}.json')
    if os.path.exists(cpath):
        with open(cpath, 'r', encoding='utf-8') as f:
            print(f'  [CACHE HIT] {exp_key}')
            return json.load(f)
    res = fn()
    with open(cpath, 'w', encoding='utf-8') as f:
        json.dump(res, f, indent=2)
    return res

def run_single_training(exp_key, base_model_or_path, train_sub_df, val_sub_df, text_col, eval_text_col,
                        mode='full', r=8, alpha=32, lr=2e-5, epochs=3, batch_size=16, max_len=128, seed=42,
                        save_model_dir=None, tokenizer_id=None):
    def _exec():
        set_all_seeds(seed)
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        tok = AutoTokenizer.from_pretrained(tokenizer_id or base_model_or_path)
        model = AutoModelForSequenceClassification.from_pretrained(base_model_or_path, num_labels=3)
        if mode == 'head_only':
            for p in model.parameters(): p.requires_grad = False
            for n, p in model.named_parameters():
                if 'classifier' in n or 'score' in n: p.requires_grad = True
        elif mode == 'lora':
            lcfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=r, lora_alpha=alpha, lora_dropout=0.1, target_modules=['query', 'value'], bias='none')
            model = get_peft_model(model, lcfg)
        model = model.to(device)
        pinfo = count_param_breakdown(model)
        cw_tensor, cw_list = compute_class_weights(train_sub_df)
        tr_ds = TextDataset(train_sub_df[text_col].tolist(), train_sub_df['label'].tolist(), tok, max_len=max_len)
        va_ds = TextDataset(val_sub_df[text_col].tolist(), val_sub_df['label'].tolist(), tok, max_len=max_len)
        ckpt_out = os.path.join(LOCAL_CKPT_DIR, f'{exp_key}_{int(time.time())}')
        args = build_training_args(ckpt_out, num_epochs=epochs, batch_size=batch_size, lr=lr, seed=seed, save_best=True)
        trainer = ClassWeightedTrainer(model=model, args=args, train_dataset=tr_ds, eval_dataset=va_ds,
                                       tokenizer=tok, data_collator=DataCollatorWithPadding(tok),
                                       compute_metrics=eval_metrics_fn, class_weights=cw_tensor)
        tr_out = trainer.train()
        elapsed = time.time() - t0
        vram = torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0.0
        steps = int(trainer.state.global_step)
        ms_per_step = (elapsed * 1000.0) / max(1, steps)
        if save_model_dir:
            os.makedirs(save_model_dir, exist_ok=True)
            trainer.save_model(save_model_dir)
            tok.save_pretrained(save_model_dir)
        eval_res, _ = evaluate_test_comprehensive(trainer, tok, eval_text_col, max_len=max_len, pred_tag=exp_key)
        del model, trainer
        shutil.rmtree(ckpt_out, ignore_errors=True)
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()
        return {
            'exp_key': exp_key,
            'mode': mode,
            'seed': seed,
            'lr': lr,
            'epochs': epochs,
            'train_n': int(len(train_sub_df)),
            'steps': steps,
            'train_time_sec': round(elapsed, 2),
            'ms_per_step': round(ms_per_step, 2),
            'peak_vram_mb': round(vram, 2),
            'class_weights': [round(x, 4) for x in cw_list],
            'params': pinfo,
            'eval': eval_res
        }
    # Note: if save_model_dir is needed downstream (Stage 1) and doesn't exist yet on fresh runtime, force rerun
    cpath = os.path.join(CACHE_DIR, f'{exp_key}.json')
    if os.path.exists(cpath) and (save_model_dir is None or os.path.exists(save_model_dir)):
        if os.path.exists(os.path.join(PREDS_DIR, f'{exp_key}.npy')):
            with open(cpath, 'r', encoding='utf-8') as f:
                print(f'  [CACHE HIT] {exp_key}')
                return json.load(f)
    res = _exec()
    with open(cpath, 'w', encoding='utf-8') as f: json.dump(res, f, indent=2)
    print(f'  [DONE] {exp_key} | Pooled Acc={res["eval"]["pooled"]["accuracy"]*100:.2f}% | MacroF1={res["eval"]["pooled"]["macro_f1"]:.4f} | Time={res["train_time_sec"]}s | VRAM={res["peak_vram_mb"]}MB')
    return res

print('Universal harness & caching system initialized.')

Universal harness & caching system initialized.


In [ ]:
# Cell 5: Majority Baseline + Stage 1 (Standard) & Stage 2 (Per-Dialect Zero-Shot) + Normalizer Ablation
def compute_majority_baseline():
    maj_label = int(train_df['label'].value_counts().idxmax())
    maj_preds = np.full(len(test_df), maj_label, dtype=int)
    y_true = test_df['label'].values
    res = {'majority_label': ID_TO_LABEL[maj_label], 'pooled': full_slice_report(y_true, maj_preds), 'per_dialect': {}}
    for reg in REGIONS:
        m = (test_df['Dialect_Region'].values == reg)
        res['per_dialect'][reg] = full_slice_report(y_true[m], maj_preds[m])
    return res

majority_res = run_cached('baseline_majority', compute_majority_baseline)
print(f'Majority Baseline (Pred={majority_res["majority_label"]}): Acc={majority_res["pooled"]["accuracy"]*100:.2f}% | F1={majority_res["pooled"]["macro_f1"]:.4f}')

stage1_stage2_results = {}
for m_key, hf_id in BENCHMARK_MODELS.items():
    t_col = 'Standard_Text_BUET' if m_key == 'BanglaBERT' else 'Standard_Text'
    d_col = 'Dialect_Text_BUET'  if m_key == 'BanglaBERT' else 'Dialect_Text'
    stage1_stage2_results[m_key] = {}
    for sd in SEEDS:
        s1_dir = os.path.join(LOCAL_S1_DIR, f's1_{m_key.lower()}_s{sd}')
        # Run Stage 1 (evaluates on Standard_Text test AND saves model)
        s1_res = run_single_training(f'stage1_{m_key.lower()}_s{sd}', hf_id, train_df, val_df,
                                     t_col, t_col, mode='full', lr=2e-5, epochs=3, seed=sd, save_model_dir=s1_dir)
        # Evaluate Stage 2 Zero-Shot on Dialect_Text using saved Stage 1 model
        def _eval_s2(s1_dir=s1_dir, d_col=d_col, sd=sd, m_key=m_key):
            tok = AutoTokenizer.from_pretrained(s1_dir)
            mod = AutoModelForSequenceClassification.from_pretrained(s1_dir).to(device)
            tr = ClassWeightedTrainer(model=mod, tokenizer=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=eval_metrics_fn)
            s2_eval, _ = evaluate_test_comprehensive(tr, tok, d_col, max_len=128, pred_tag=f'stage2_{m_key.lower()}_s{sd}')
            del mod, tr; torch.cuda.empty_cache(); gc.collect()
            return {'eval': s2_eval}
        s2_key = f'stage2_{m_key.lower()}_s{sd}'
        s2_cpath = os.path.join(CACHE_DIR, f'{s2_key}.json')
        if os.path.exists(s2_cpath) and os.path.exists(os.path.join(PREDS_DIR, f'{s2_key}.npy')):
            with open(s2_cpath, 'r', encoding='utf-8') as f: s2_res = json.load(f)
            print(f'  [CACHE HIT] {s2_key}')
        else:
            s2_res = _eval_s2()
            with open(s2_cpath, 'w', encoding='utf-8') as f: json.dump(s2_res, f, indent=2)
            print(f'  [DONE] {s2_key} | Zero-Shot Pooled Acc={s2_res["eval"]["pooled"]["accuracy"]*100:.2f}% | MacroF1={s2_res["eval"]["pooled"]["macro_f1"]:.4f}')
        stage1_stage2_results[m_key][f'seed_{sd}'] = {'stage1': s1_res, 'stage2': s2_res}

# BanglaBERT Preprocessing Intervention: Raw Text (unnormalized) vs BUET Normalized
raw_s1_dir = os.path.join(LOCAL_S1_DIR, 's1_banglabert_raw_s42')
bb_raw_s1 = run_single_training('stage1_banglabert_raw_s42', 'csebuetnlp/banglabert', train_df, val_df,
                                'Standard_Text_Raw', 'Standard_Text_Raw', mode='full', lr=2e-5, epochs=3, seed=42, save_model_dir=raw_s1_dir)
def _eval_s2_raw():
    tok = AutoTokenizer.from_pretrained(raw_s1_dir)
    mod = AutoModelForSequenceClassification.from_pretrained(raw_s1_dir).to(device)
    tr = ClassWeightedTrainer(model=mod, tokenizer=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=eval_metrics_fn)
    ev, _ = evaluate_test_comprehensive(tr, tok, 'Dialect_Text_Raw', max_len=128, pred_tag='stage2_banglabert_raw_s42')
    del mod, tr; torch.cuda.empty_cache(); gc.collect()
    return {'eval': ev}
bb_raw_s2 = run_cached('stage2_banglabert_raw_s42', _eval_s2_raw)
print('Stage 1 & Stage 2 multi-seed evaluations complete.')

  [CACHE HIT] baseline_majority
Majority Baseline (Pred=negative): Acc=62.43% | F1=0.2562


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

... (95 outputs left out)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.391726,0.542129,0.815867,0.763810
2,0.253835,0.651870,0.819420,0.770434
3,0.129397,0.845066,0.824156,0.771268


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  [DONE] stage1_banglabert_raw_s42 | Pooled Acc=82.97% | MacroF1=0.7782 | Time=115.7s | VRAM=1837.7MB
  [CACHE HIT] stage2_banglabert_raw_s42
Stage 1 & Stage 2 multi-seed evaluations complete.


In [ ]:
# Cell 6: Stage 3 Full Fine-Tuning Upper Bounds & Tokenizer-Free CANINE-C Intervention
stage3_results = {'pooled_full_ft': {}, 'anchor_chattogram_full_ft': {}, 'canine_c': {}}
tr_chat = train_df[train_df['Dialect_Region'] == 'Chattogram'].reset_index(drop=True)
va_chat = val_df[val_df['Dialect_Region'] == 'Chattogram'].reset_index(drop=True)

# 1. BanglaBERT Full-FT Pooled (3 seeds) & Chattogram-Only Full-FT (3 seeds)
for sd in SEEDS:
    stage3_results['pooled_full_ft'][f'BanglaBERT_s{sd}'] = run_single_training(
        f's3_pooled_banglabert_s{sd}', 'csebuetnlp/banglabert', train_df, val_df,
        'Dialect_Text_BUET', 'Dialect_Text_BUET', mode='full', lr=2e-5, epochs=3, seed=sd)
    stage3_results['anchor_chattogram_full_ft'][f'BanglaBERT_s{sd}'] = run_single_training(
        f's3_chat_full_banglabert_s{sd}', 'csebuetnlp/banglabert', tr_chat, va_chat,
        'Dialect_Text_BUET', 'Dialect_Text_BUET', mode='full', lr=2e-5, epochs=5, seed=sd)

# 2. mBERT & XLM-R Full-FT (Pooled + Chattogram Anchor-Only; solves XLM-R baseline gap)
for m_key in ['mBERT', 'XLM-RoBERTa']:
    hf_id = BENCHMARK_MODELS[m_key]
    stage3_results['pooled_full_ft'][f'{m_key}_s42'] = run_single_training(
        f's3_pooled_{m_key.lower()}_s42', hf_id, train_df, val_df,
        'Dialect_Text', 'Dialect_Text', mode='full', lr=2e-5, epochs=3, seed=42)
    stage3_results['anchor_chattogram_full_ft'][f'{m_key}_s42'] = run_single_training(
        f's3_chat_full_{m_key.lower()}_s42', hf_id, tr_chat, va_chat,
        'Dialect_Text', 'Dialect_Text', mode='full', lr=2e-5, epochs=5, seed=42)

# 3. Causal Tokenization Baseline: google/canine-c (Tokenizer-Free Unicode Codepoint Encoder)
try:
    stage3_results['canine_c']['pooled_s42'] = run_single_training(
        's3_pooled_canine_s42', 'google/canine-c', train_df, val_df,
        'Dialect_Text', 'Dialect_Text', mode='full', lr=3e-5, epochs=4, batch_size=16, max_len=256, seed=42)
    stage3_results['canine_c']['chattogram_s42'] = run_single_training(
        's3_chat_canine_s42', 'google/canine-c', tr_chat, va_chat,
        'Dialect_Text', 'Dialect_Text', mode='full', lr=3e-5, epochs=5, batch_size=16, max_len=256, seed=42)
except Exception as e:
    print(f'CANINE-C run note: {e}')
    stage3_results['canine_c']['error'] = str(e)

print('Stage 3 Full-FT & CANINE-C experiments complete.')

  [CACHE HIT] s3_pooled_banglabert_s42
  [CACHE HIT] s3_chat_full_banglabert_s42
  [CACHE HIT] s3_pooled_banglabert_s101
  [CACHE HIT] s3_chat_full_banglabert_s101
  [CACHE HIT] s3_pooled_banglabert_s2026
  [CACHE HIT] s3_chat_full_banglabert_s2026
  [CACHE HIT] s3_pooled_mbert_s42
  [CACHE HIT] s3_chat_full_mbert_s42
  [CACHE HIT] s3_pooled_xlm-roberta_s42
  [CACHE HIT] s3_chat_full_xlm-roberta_s42
  [CACHE HIT] s3_pooled_canine_s42
  [CACHE HIT] s3_chat_canine_s42
Stage 3 Full-FT & CANINE-C experiments complete.


In [9]:
# Cell 7: Stage 4 Fair PEFT Ablations (Head-Only, Raw->LoRA, Stage1->LoRA, Pooled LoRA, XLM-R Fix)
peft_controls = {}

# A. BanglaBERT (3 seeds): Compare Head-Only vs LoRA from BOTH Raw Pretrained AND Stage1 Fine-Tuned
for sd in SEEDS:
    s1_bb_dir = os.path.join(LOCAL_S1_DIR, f's1_banglabert_s{sd}')
    # 1. Head-Only from Raw
    peft_controls[f'bb_raw_head_only_s{sd}'] = run_single_training(
        f's4_bb_raw_head_s{sd}', 'csebuetnlp/banglabert', tr_chat, va_chat,
        'Dialect_Text_BUET', 'Dialect_Text_BUET', mode='head_only', lr=5e-4, epochs=5, seed=sd)
    # 2. Head-Only from Stage 1
    peft_controls[f'bb_s1_head_only_s{sd}'] = run_single_training(
        f's4_bb_s1_head_s{sd}', s1_bb_dir, tr_chat, va_chat,
        'Dialect_Text_BUET', 'Dialect_Text_BUET', mode='head_only', lr=5e-4, epochs=5, seed=sd,
        tokenizer_id='csebuetnlp/banglabert')
    # 3. Anchor LoRA from Raw (matches old code paradigm on clean split)
    peft_controls[f'bb_raw_lora_s{sd}'] = run_single_training(
        f's4_bb_raw_lora_s{sd}', 'csebuetnlp/banglabert', tr_chat, va_chat,
        'Dialect_Text_BUET', 'Dialect_Text_BUET', mode='lora', r=8, alpha=32, lr=5e-4, epochs=5, seed=sd)
    # 4. Anchor LoRA from Stage 1 (adapts Standard classifier to Dialect)
    peft_controls[f'bb_s1_lora_s{sd}'] = run_single_training(
        f's4_bb_s1_lora_s{sd}', s1_bb_dir, tr_chat, va_chat,
        'Dialect_Text_BUET', 'Dialect_Text_BUET', mode='lora', r=8, alpha=32, lr=3e-4, epochs=5, seed=sd,
        tokenizer_id='csebuetnlp/banglabert')
    # 5. Pooled Multi-Dialect LoRA from Stage 1
    peft_controls[f'bb_s1_pooled_lora_s{sd}'] = run_single_training(
        f's4_bb_s1_pooled_lora_s{sd}', s1_bb_dir, train_df, val_df,
        'Dialect_Text_BUET', 'Dialect_Text_BUET', mode='lora', r=8, alpha=32, lr=3e-4, epochs=5, seed=sd,
        tokenizer_id='csebuetnlp/banglabert')

# B. mBERT Anchor LoRA (Raw->LoRA & Stage1->LoRA, 3 seeds)
for sd in SEEDS:
    s1_mb_dir = os.path.join(LOCAL_S1_DIR, f's1_mbert_s{sd}')
    peft_controls[f'mb_raw_lora_s{sd}'] = run_single_training(
        f's4_mb_raw_lora_s{sd}', 'bert-base-multilingual-cased', tr_chat, va_chat,
        'Dialect_Text', 'Dialect_Text', mode='lora', r=8, alpha=32, lr=5e-4, epochs=5, seed=sd)
    peft_controls[f'mb_s1_lora_s{sd}'] = run_single_training(
        f's4_mb_s1_lora_s{sd}', s1_mb_dir, tr_chat, va_chat,
        'Dialect_Text', 'Dialect_Text', mode='lora', r=8, alpha=32, lr=3e-4, epochs=5, seed=sd,
        tokenizer_id='bert-base-multilingual-cased')

# C. XLM-RoBERTa Optimization Collapse Diagnosis (Tuned LRs & Ranks + Stage 1 Init)
s1_xr_dir = os.path.join(LOCAL_S1_DIR, 's1_xlm-roberta_s42')
xlmr_sweep = [
    ('xr_raw_r8_lr5e4_repro', 'xlm-roberta-base', 8, 32, 5e-4),
    ('xr_raw_r8_lr1e4',       'xlm-roberta-base', 8, 16, 1e-4),
    ('xr_raw_r16_lr2e4',      'xlm-roberta-base', 16, 32, 2e-4),
    ('xr_s1_r8_lr1e4',        s1_xr_dir,          8, 16, 1e-4),
    ('xr_s1_r32_lr1e4',       s1_xr_dir,         32, 64, 1e-4),
]
for tag, init_p, rank_v, alpha_v, lr_v in xlmr_sweep:
    peft_controls[tag] = run_single_training(
        f's4_{tag}_s42', init_p, tr_chat, va_chat, 'Dialect_Text', 'Dialect_Text',
        mode='lora', r=rank_v, alpha=alpha_v, lr=lr_v, epochs=5, seed=42, tokenizer_id='xlm-roberta-base')

print('Stage 4 PEFT Controls & XLM-R optimization sweep complete.')

  [CACHE HIT] s4_bb_raw_head_s42
  [CACHE HIT] s4_bb_s1_head_s42
  [CACHE HIT] s4_bb_raw_lora_s42
  [CACHE HIT] s4_bb_s1_lora_s42
  [CACHE HIT] s4_bb_s1_pooled_lora_s42
  [CACHE HIT] s4_bb_raw_head_s101
  [CACHE HIT] s4_bb_s1_head_s101
  [CACHE HIT] s4_bb_raw_lora_s101
  [CACHE HIT] s4_bb_s1_lora_s101
  [CACHE HIT] s4_bb_s1_pooled_lora_s101
  [CACHE HIT] s4_bb_raw_head_s2026
  [CACHE HIT] s4_bb_s1_head_s2026
  [CACHE HIT] s4_bb_raw_lora_s2026
  [CACHE HIT] s4_bb_s1_lora_s2026
  [CACHE HIT] s4_bb_s1_pooled_lora_s2026
  [CACHE HIT] s4_mb_raw_lora_s42
  [CACHE HIT] s4_mb_s1_lora_s42
  [CACHE HIT] s4_mb_raw_lora_s101
  [CACHE HIT] s4_mb_s1_lora_s101


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.974099,0.883355,0.592934,0.560308
2,0.915762,0.803845,0.623656,0.603685
3,0.791177,0.754912,0.683564,0.657917
4,0.699002,0.780707,0.654378,0.632637
5,0.684218,0.770000,0.662058,0.639046


  [DONE] s4_mb_raw_lora_s2026 | Pooled Acc=66.11% | MacroF1=0.5825 | Time=92.51s | VRAM=1724.97MB


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

... (20 outputs left out)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.828745,0.795434,0.672811,0.653022
2,0.789241,0.759424,0.675883,0.653142
3,0.723908,0.765559,0.686636,0.663580
4,0.683926,0.768350,0.692780,0.671195
5,0.644487,0.770743,0.694316,0.659172


  [DONE] s4_xr_s1_r32_lr1e4_s42 | Pooled Acc=72.88% | MacroF1=0.6746 | Time=86.31s | VRAM=2021.51MB
Stage 4 PEFT Controls & XLM-R optimization sweep complete.


In [10]:
# Cell 8: Bidirectional 4x4 Anchor Transfer Matrix (Train on each Dialect -> Zero-Shot on all 4)
transfer_4x4_s1_lora = {r: {} for r in REGIONS}
transfer_4x4_raw_lora = {r: {} for r in REGIONS}

for anchor_reg in REGIONS:
    tr_anc = train_df[train_df['Dialect_Region'] == anchor_reg].reset_index(drop=True)
    va_anc = val_df[val_df['Dialect_Region'] == anchor_reg].reset_index(drop=True)
    for sd in SEEDS:
        if anchor_reg == 'Chattogram':
            transfer_4x4_s1_lora[anchor_reg][f's{sd}'] = peft_controls[f'bb_s1_lora_s{sd}']
            transfer_4x4_raw_lora[anchor_reg][f's{sd}'] = peft_controls[f'bb_raw_lora_s{sd}']
        else:
            s1_bb_dir = os.path.join(LOCAL_S1_DIR, f's1_banglabert_s{sd}')
            transfer_4x4_s1_lora[anchor_reg][f's{sd}'] = run_single_training(
                f'matrix_s1_lora_{anchor_reg.lower()}_s{sd}', s1_bb_dir, tr_anc, va_anc,
                'Dialect_Text_BUET', 'Dialect_Text_BUET', mode='lora', r=8, alpha=32, lr=3e-4, epochs=5, seed=sd,
                tokenizer_id='csebuetnlp/banglabert')
            transfer_4x4_raw_lora[anchor_reg][f's{sd}'] = run_single_training(
                f'matrix_raw_lora_{anchor_reg.lower()}_s{sd}', 'csebuetnlp/banglabert', tr_anc, va_anc,
                'Dialect_Text_BUET', 'Dialect_Text_BUET', mode='lora', r=8, alpha=32, lr=5e-4, epochs=5, seed=sd)

# Matched-Data Hardware Benchmark (exact same Chattogram Anchor data & 5 epochs)
hw_matched = {
    'Full_FT_Chattogram_5ep':    stage3_results['anchor_chattogram_full_ft']['BanglaBERT_s42'],
    'Anchor_LoRA_r8_5ep':        peft_controls['bb_s1_lora_s42'],
    'Head_Only_Chattogram_5ep':  peft_controls['bb_s1_head_only_s42']
}
print('\nMatched-Data Hardware Efficiency (Chattogram n=5,427, 5 epochs):')
for k, v in hw_matched.items():
    print(f'  {k:<26}: VRAM={v["peak_vram_mb"]:.1f}MB | Time={v["train_time_sec"]}s | ms/step={v["ms_per_step"]} | TrainParams={v["params"]["trainable_total"]:,} ({v["params"]["trainable_pct"]}%)')

# McNemar Tests (Stage 1->Chattogram Anchor LoRA vs Stage 2 Zero-Shot, seed=42)
y_true_all = test_df['label'].values
p_s2 = np.load(os.path.join(PREDS_DIR, 'stage2_banglabert_s42.npy'))
p_lora = np.load(os.path.join(PREDS_DIR, 's4_bb_s1_lora_s42.npy'))
p_raw_lora = np.load(os.path.join(PREDS_DIR, 's4_bb_raw_lora_s42.npy'))
mcnemar_summary = {'pooled': mcnemar_vs(y_true_all, p_s2, p_lora), 'per_dialect': {}}
for reg in REGIONS:
    m = (test_df['Dialect_Region'].values == reg)
    mcnemar_summary['per_dialect'][reg] = mcnemar_vs(y_true_all[m], p_s2[m], p_lora[m])
print('\nMcNemar Test (Stage 2 Zero-Shot -> Stage 1+Chattogram Anchor LoRA, seed=42):')
print(json.dumps(mcnemar_summary, indent=2))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.407255,0.690741,0.852601,0.796376
2,0.342020,0.733357,0.872832,0.817938
3,0.276566,0.698088,0.858382,0.809390
4,0.251902,0.770366,0.881503,0.826214
5,0.199379,0.762891,0.881503,0.823334


  [DONE] matrix_s1_lora_sylhet_s42 | Pooled Acc=77.45% | MacroF1=0.7145 | Time=45.04s | VRAM=989.68MB


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr

... (63 outputs left out)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.667383,0.541853,0.841040,0.761873
2,0.463104,0.572666,0.815029,0.768325
3,0.347011,0.533341,0.852601,0.796949
4,0.254536,0.509942,0.881503,0.826646
5,0.224607,0.532749,0.878613,0.826747


  [DONE] matrix_raw_lora_mymensingh_s2026 | Pooled Acc=74.24% | MacroF1=0.6496 | Time=44.35s | VRAM=1004.42MB

Matched-Data Hardware Efficiency (Chattogram n=5,427, 5 epochs):
  Full_FT_Chattogram_5ep    : VRAM=2190.6MB | Time=85.26s | ms/step=52.31 | TrainParams=110,619,651 (100.0%)
  Anchor_LoRA_r8_5ep        : VRAM=1174.0MB | Time=82.21s | ms/step=50.43 | TrainParams=887,811 (0.7962%)
  Head_Only_Chattogram_5ep  : VRAM=523.3MB | Time=43.76s | ms/step=26.85 | TrainParams=592,899 (0.536%)

McNemar Test (Stage 2 Zero-Shot -> Stage 1+Chattogram Anchor LoRA, seed=42):
{
  "pooled": {
    "b_fixes_a": 203,
    "b_breaks_a": 115,
    "net_gain": 88,
    "p_value": 1e-06
  },
  "per_dialect": {
    "Chattogram": {
      "b_fixes_a": 142,
      "b_breaks_a": 71,
      "net_gain": 71,
      "p_value": 1e-06
    },
    "Sylhet": {
      "b_fixes_a": 21,
      "b_breaks_a": 17,
      "net_gain": 4,
      "p_value": 0.627103
    },
    "Noakhali": {
      "b_fixes_a": 20,
      "b_breaks_a": 13,

In [11]:
# Cell 9: Sentence-Level Statistical Modeling, 200-Error Audit Export & Master JSON
import statsmodels.api as sm
import statsmodels.formula.api as smf

utt_test = utt_df[utt_df['split'] == 'test'].reset_index(drop=True).copy()
y_true_all = test_df['label'].values
p_s2 = np.load(os.path.join(PREDS_DIR, 'stage2_banglabert_s42.npy'))
p_s1_lora = np.load(os.path.join(PREDS_DIR, 's4_bb_s1_lora_s42.npy'))
p_raw_lora = np.load(os.path.join(PREDS_DIR, 's4_bb_raw_lora_s42.npy'))
p_full_pooled = np.load(os.path.join(PREDS_DIR, 's3_pooled_banglabert_s42.npy'))

utt_test['err_stage2'] = (p_s2 != y_true_all).astype(int)
utt_test['err_s1_lora'] = (p_s1_lora != y_true_all).astype(int)
utt_test['err_raw_lora'] = (p_raw_lora != y_true_all).astype(int)
utt_test['err_full_ft'] = (p_full_pooled != y_true_all).astype(int)

# Sentence-level Logistic Regression controlling for edit distance, word count, region, source, sentiment
logit_formula = 'err_stage2 ~ fert_bb_buet + norm_edit_dist + word_count + C(Dialect_Region) + C(Source) + C(Sentiment)'
logit_res = {}
try:
    try:
        mod_logit = smf.logit(logit_formula, data=utt_test).fit(disp=False, cov_type='HC1')
    except Exception:
        mod_logit = smf.logit(logit_formula, data=utt_test).fit_regularized(alpha=0.1, disp=False)
    print('\n=== SENTENCE-LEVEL LOGISTIC REGRESSION ON STAGE 2 ERROR (n={}) ==='.format(len(utt_test)))
    try: print(mod_logit.summary2().tables[1][['Coef.', 'Std.Err.', 'z', 'P>|z|']])
    except Exception: print(mod_logit.params)
    for term in mod_logit.params.index:
        coef = float(mod_logit.params[term])
        pval = float(mod_logit.pvalues[term])
        logit_res[term] = {'coef': round(coef, 4), 'odds_ratio': round(float(np.exp(coef)), 4), 'p_value': float(f'{pval:.3e}')}
except Exception as e:
    print(f'Logit fit note: {e}')
    logit_res['error'] = str(e)

# Dialect-level correlations across Stage 2 Zero-Shot
from scipy.stats import pearsonr, spearmanr
s2_err_by_reg = [100.0 * (1.0 - stage1_stage2_results['BanglaBERT']['seed_42']['stage2']['eval']['per_dialect'][r]['accuracy']) for r in REGIONS]
deltas_reg = [tokenizer_summary['buet_norm']['BanglaBERT'][r]['corpus_delta_pct'] for r in REGIONS]
r_4, p_4 = pearsonr(deltas_reg, s2_err_by_reg)
print(f'\n4-Dialect Pearson r (BUET delta vs Stage2 Err): r={r_4:.4f}, R^2={r_4**2:.4f}, p={p_4:.4f}')

# Export 200 Stratified Stage 2 Misclassified Test Samples for Human Error Taxonomy Audit
tok_bb = AutoTokenizer.from_pretrained('csebuetnlp/banglabert')
err_mask = (p_s2 != y_true_all)
err_df = test_df[err_mask].copy()
err_df['True_Label'] = [ID_TO_LABEL[i] for i in y_true_all[err_mask]]
err_df['Stage2_Pred'] = [ID_TO_LABEL[i] for i in p_s2[err_mask]]
err_df['LoRA_Pred'] = [ID_TO_LABEL[i] for i in p_s1_lora[err_mask]]
err_df['LoRA_Fixed'] = (p_s1_lora[err_mask] == y_true_all[err_mask])
err_df['Subword_Ratio'] = utt_test['fert_bb_buet'].values[err_mask]
err_df['BanglaBERT_Subwords'] = [' '.join(tok_bb.tokenize(str(t))) for t in err_df['Dialect_Text_BUET']]

sampled_parts = []
for reg in REGIONS:
    sub_r = err_df[err_df['Dialect_Region'] == reg]
    take_n = min(50, len(sub_r))
    sampled_parts.append(sub_r.sample(n=take_n, random_state=42))
audit_200 = pd.concat(sampled_parts).reset_index(drop=True)
if len(audit_200) < 200 and len(err_df) > len(audit_200):
    rem_pool = err_df[~err_df['sample_id'].isin(set(audit_200['sample_id']))]
    extra = rem_pool.sample(n=min(200 - len(audit_200), len(rem_pool)), random_state=42)
    audit_200 = pd.concat([audit_200, extra]).reset_index(drop=True)
audit_200['Error_Category'] = ''
cols_out = ['sample_id', 'Dialect_Region', 'Source', 'Standard_Text', 'Dialect_Text', 'True_Label', 'Stage2_Pred', 'LoRA_Pred', 'LoRA_Fixed', 'Subword_Ratio', 'BanglaBERT_Subwords', 'Error_Category']
err_csv_path = os.path.join(OUT_DIR, 'error_audit_candidates_200.csv')
audit_200[cols_out].to_csv(err_csv_path, index=False, encoding='utf-8-sig')
print(f'Exported {len(audit_200)} real Stage 2 misclassifications to: {err_csv_path}')

# Save Consolidated Master V2 Results JSON
def agg_3seeds(runs_list):
    accs = [r['eval']['pooled']['accuracy'] for r in runs_list]
    f1s  = [r['eval']['pooled']['macro_f1'] for r in runs_list]
    return {'acc_mean': round(float(np.mean(accs)), 4), 'acc_sd': round(float(np.std(accs, ddof=1)), 4),
            'f1_mean': round(float(np.mean(f1s)), 4),  'f1_sd': round(float(np.std(f1s, ddof=1)), 4)}

MASTER_V2 = {
    'metadata': {'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'), 'seeds': SEEDS, 'clean_total': len(df), 'test_n': len(test_df)},
    'tokenizer_diagnostics': tokenizer_summary,
    'majority_baseline': majority_res,
    'stage1_stage2_multi_seed': stage1_stage2_results,
    'normalizer_intervention_raw_vs_buet': {'raw_s1': bb_raw_s1, 'raw_s2': bb_raw_s2},
    'stage3_upper_bounds_and_canine': stage3_results,
    'stage4_peft_controls_and_xlmr_sweep': peft_controls,
    'transfer_4x4_stage1_lora': transfer_4x4_s1_lora,
    'transfer_4x4_raw_lora': transfer_4x4_raw_lora,
    'mcnemar_s2_vs_s1_lora': mcnemar_summary,
    'matched_hardware': hw_matched,
    'sentence_level_logit': logit_res,
    'dialect_4pt_corr': {'r': round(float(r_4), 4), 'r2': round(float(r_4**2), 4), 'p': round(float(p_4), 4)},
    'multi_seed_summaries': {
        'bb_stage1': agg_3seeds([stage1_stage2_results['BanglaBERT'][f'seed_{s}']['stage1'] for s in SEEDS]),
        'bb_stage2': agg_3seeds([stage1_stage2_results['BanglaBERT'][f'seed_{s}']['stage2'] for s in SEEDS]),
        'bb_stage3_pooled': agg_3seeds([stage3_results['pooled_full_ft'][f'BanglaBERT_s{s}'] for s in SEEDS]),
        'bb_stage3_chat': agg_3seeds([stage3_results['anchor_chattogram_full_ft'][f'BanglaBERT_s{s}'] for s in SEEDS]),
        'bb_raw_head': agg_3seeds([peft_controls[f'bb_raw_head_only_s{s}'] for s in SEEDS]),
        'bb_s1_head': agg_3seeds([peft_controls[f'bb_s1_head_only_s{s}'] for s in SEEDS]),
        'bb_raw_lora': agg_3seeds([peft_controls[f'bb_raw_lora_s{s}'] for s in SEEDS]),
        'bb_s1_lora': agg_3seeds([peft_controls[f'bb_s1_lora_s{s}'] for s in SEEDS]),
        'bb_s1_pooled_lora': agg_3seeds([peft_controls[f'bb_s1_pooled_lora_s{s}'] for s in SEEDS]),
    }
}
master_v2_path = os.path.join(OUT_DIR, 'MASTER_TALLIP_V2_RESULTS.json')
with open(master_v2_path, 'w', encoding='utf-8') as f: json.dump(MASTER_V2, f, indent=2)
print(f'\nALL V2 EXPERIMENTS SAVED TO: {master_v2_path}')
print('\n=== 3-SEED POOLED SUMMARY (BANGLABERT) ===')
for k, v in MASTER_V2['multi_seed_summaries'].items():
    print(f'  {k:<20}: Acc={v["acc_mean"]*100:.2f}% (+/-{v["acc_sd"]*100:.2f}) | MacroF1={v["f1_mean"]:.4f} (+/-{v["f1_sd"]:.4f})')


=== SENTENCE-LEVEL LOGISTIC REGRESSION ON STAGE 2 ERROR (n=1685) ===
                                             Coef.  Std.Err.          z  \
Intercept                                -4.610174  0.580896  -7.936316   
C(Dialect_Region)[T.Mymensingh]          -0.060823  0.334769  -0.181686   
C(Dialect_Region)[T.Noakhali]            -0.218235  0.298291  -0.731617   
C(Dialect_Region)[T.Sylhet]              -0.145685  0.301612  -0.483022   
C(Source)[T.Original_Chattogram_Dataset]  0.892129  0.261548   3.410960   
C(Source)[T.SAPTAK_Manual_Annotation]     0.894077  0.255432   3.500254   
C(Sentiment)[T.neutral]                   2.928336  0.239332  12.235464   
C(Sentiment)[T.positive]                  2.997389  0.225536  13.290070   
fert_bb_buet                              0.466111  0.227106   2.052395   
norm_edit_dist                            1.984501  0.515519   3.849524   
word_count                                0.003665  0.016344   0.224243   

                             